In [1]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import sys
from astropy.io import fits


f = h5py.File('/Volumes/External SSD 512gb/dset_with_labels.h5', 'r')
list(f.keys())

['classes',
 'data',
 'flag_map',
 'freq_range',
 'labels',
 'rfi_corrected',
 'time_range',
 'timestamps']

By seeing the shape of the labels key, we can see that there are only 2854 labels for the entire dataset. Intially, I thought the entire 8K dataset was labelled but turns out only 2854 images have labels. 

Check key in the data

In [2]:
# Open the H5 file in read mode
with h5py.File('/Volumes/External SSD 512gb/dset_with_labels.h5', 'r') as file:
    print("Keys: %s" % file.keys())
    a_group_key = list(file.keys())[4]  # See the data of [index] key

    # Getting the data
    data = file[a_group_key][:]
    print(data)
    print(f"Shape of key '{a_group_key}': {data.shape}")
    print (f"Data type of key '{a_group_key}': {data.dtype}")

Keys: <KeysViewHDF5 ['classes', 'data', 'flag_map', 'freq_range', 'labels', 'rfi_corrected', 'time_range', 'timestamps']>
[1 1 2 ... 1 3 1]
Shape of key 'labels': (2854,)
Data type of key 'labels': int64


Checka specific date in the data

In [3]:
import h5py

with h5py.File('/Volumes/External SSD 512gb/dset_with_labels.h5', 'r') as file:
    timestamps = file['timestamps'][:]  # key index 7
    # Decode byte strings and search for the date
    matches = [i for i, ts in enumerate(timestamps) if b'2024-05-14' in ts]
    print(f"Indices with date 2022-07-02: {matches}")
    # Optionally, print the matching timestamps
    for i in matches:
        print(timestamps[i].decode())

Indices with date 2022-07-02: []


In [4]:
dset = f['time_range']
dset.shape

(8147, 2)

Oldest and Latest data entry

In [5]:
import h5py
import re
from datetime import datetime

# Open the H5 file in read mode
with h5py.File('/Volumes/External SSD 512gb/dset_with_labels.h5', 'r') as file:
    timestamps = file['timestamps'][:]  # Load all timestamps
    
    # Initialize variables to track oldest and latest dates
    oldest_date = None
    latest_date = None
    oldest_index = None
    latest_index = None
    
    # Regular expression to extract date part (YYYY-MM-DD)
    date_pattern = re.compile(r'(\d{4}-\d{2}-\d{2})')
    
    for i, ts in enumerate(timestamps):
        ts_decoded = ts.decode()
        match = date_pattern.search(ts_decoded)
        if match:
            # Extract the date string and convert to datetime object
            date_str = match.group(1)
            date_obj = datetime.strptime(date_str, '%Y-%m-%d')
            
            # Update oldest date if this date is older (or first one)
            if oldest_date is None or date_obj < oldest_date:
                oldest_date = date_obj
                oldest_index = i
                oldest_timestamp = ts_decoded
            
            # Update latest date if this date is newer (or first one)
            if latest_date is None or date_obj > latest_date:
                latest_date = date_obj
                latest_index = i
                latest_timestamp = ts_decoded
    
    # Print results
    print(f"Total timestamps: {len(timestamps)}")
    print(f"Oldest date: {oldest_date.strftime('%Y-%m-%d')} (index {oldest_index})")
    print(f"Oldest timestamp: {oldest_timestamp}")
    print(f"Latest date: {latest_date.strftime('%Y-%m-%d')} (index {latest_index})")
    print(f"Latest timestamp: {latest_timestamp}")
    
    # Calculate the time span
    time_span = latest_date - oldest_date
    print(f"Dataset spans {time_span.days} days ({time_span.days/365.25:.2f} years)")
    
    # Count records per year to see distribution
    year_counts = {}
    for ts in timestamps:
        ts_decoded = ts.decode()
        match = date_pattern.search(ts_decoded)
        if match:
            year = match.group(1)[:4]  # Extract just the year
            year_counts[year] = year_counts.get(year, 0) + 1
    
    print("\nDistribution by year:")
    for year in sorted(year_counts.keys()):
        print(f"  {year}: {year_counts[year]} records")

Total timestamps: 8147
Oldest date: 2022-05-02 (index 56)
Oldest timestamp: 858918_2022-05-02_16:00:00.000000
Latest date: 2023-03-25 (index 8054)
Latest timestamp: 885608_2023-03-25_14:15:00.000000
Dataset spans 327 days (0.90 years)

Distribution by year:
  2022: 7279 records
  2023: 868 records


In [ ]:
# Open the H5 file in read mode
with h5py.File('/Volumes/External SSD 512gb/dset_with_labels.h5', 'r') as file:
    print("Keys: %s" % file.keys())
    a_group_key = list(file.keys())[3] # See the data of [index] key
    
    # Getting the data
    data = list(file[a_group_key])
    print(data)

freqset = f['freq_range']
freqset.shape    

#### Plot heatmap

In [ ]:
# Load the dataset from HDF5 file
with h5py.File('/Volumes/External SSD 512gb/dset_with_labels.h5', 'r') as f:
    dset = f['data']  # shape = (8147, 500, 800)

    # Plot heatmaps for the first 5 samples (each of shape (500, 800))
    plt.figure(figsize=(15, 10))
    for i in range(20):
        plt.subplot(5, 4, i + 1)
        sns.heatmap(dset[i].T, cmap='viridis')  # Transpose to invert rows and columns
        plt.title(f'Heatmap of Sample {i + 1}')
        plt.xlabel('Rows')
        plt.ylabel('Columns')

    plt.tight_layout()
    plt.show()

Above is all freq axis plotted with all time axis

#### Seperating out diff freq ranges, time on axis

In [6]:
# Load HDF5 dataset
with h5py.File('/Volumes/External SSD 512gb/dset_with_labels.h5', 'r') as f:
    data = f['data'][:]  # (8147, 500, 800)
    rfi_corrected = f['rfi_corrected'][:]  # (8147, 500, 800)
    flag_map = f['flag_map'][:]  # (8147, 500, 800)
    timestamps = f['timestamps'][:]  # (8147,)
    freq_range = f['freq_range'][:]  # (8147, 2)
    time_range = f['time_range'][:]  # (8147, 2)
    labels = f['labels'][:]  # (2854,)
    classes = f['classes'][:]  # (6,)


# Plot multiple frames with metadata
def plot_multiple_frames(indices, use_rfi_corrected=True):
    n = len(indices)
    plt.figure(figsize=(15, 5 * n))
    
    for i, index in enumerate(indices):
        spec_data = rfi_corrected[index] if use_rfi_corrected else data[index]
        plt.subplot(n, 1, i + 1)
        plt.imshow(spec_data, aspect='auto', origin='lower', cmap='viridis')
        plt.colorbar(label='Intensity')
        plt.title(f"Dynamic Spectrum - Frame {index}\nTime: {timestamps[index]} | Freq: {freq_range[index][0]}-{freq_range[index][1]} MHz")
        plt.xlabel('Time Samples')
        plt.ylabel('Frequency Bins')
    
    plt.tight_layout()
    plt.show()
# Plot multiple frames
plot_multiple_frames([0, 1, 2, 3, 4, 5, 6 ,7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19])
# # Plot multiple frames with RFI corrected data
# plot_multiple_frames([0, 1, 2, 3, 4], use_rfi_corrected=True)


# Plotting the labels distribution
def plot_labels():
    plt.figure(figsize=(12, 6))
    plt.bar(range(len(labels)), labels, color='blue', alpha=0.7, label='Labels')
    plt.xticks(range(len(labels)), [f'Label {i}' for i in range(len(labels))])
    plt.xlabel('Labels')
    plt.ylabel('Count')
    plt.title('Labels Distribution')
    plt.legend()
    plt.tight_layout()
    plt.show()

# Frequency of label occurrences
def plot_label_frequencies():
    label_counts = Counter(labels)
    sorted_labels = sorted(label_counts.keys())
    frequencies = [label_counts[label] for label in sorted_labels]

    plt.figure(figsize=(10, 5))
    plt.bar(sorted_labels, frequencies, color='green', alpha=0.7)
    plt.xlabel('Label Index')
    plt.ylabel('Frequency')
    plt.title('Frequency of Each Label')
    plt.xticks(sorted_labels)
    plt.tight_layout()
    plt.show()

# Plot label frequencies
plot_label_frequencies()


: 

The number of labels we have is (2854). Now things makes sense from below counts for previous implementation. The data model trained by mattia was size (2723,500,500) since that was only the amount of labels he had. Rest of the data around 5424 image data was not used since lack of labels. 

In [ ]:
# Print frequencies of labels 2 and 4
label_counts = Counter(labels)
print(f"Frequency of label 1: {label_counts[1]}")
print(f"Frequency of label 2: {label_counts[2]}")
print(f"Frequency of label 3: {label_counts[3]}")
print(f"Frequency of label 4: {label_counts[4]}")
print(f"Frequency of label 5: {label_counts[5]}")
print(f"Frequency of label 6: {label_counts[6]}")

NameError: name 'labels' is not defined

Lets compare labels to validate whats in above images.

In [ ]:
# Open the H5 file in read mode
with h5py.File('dset_with_labels.h5', 'r') as file:
    print("Keys: %s" % file.keys())
    a_group_key = list(file.keys())[7] # See the data of [index] key
    
    # Getting the data
    data = list(file[a_group_key])
    print(data)